In [1]:
import os
from dotenv import load_dotenv
import gradio as gr
from groq import Groq
from openai import OpenAI

C:\Users\91944\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [3]:
import platform

# 1. Detect the Architecture automatically
arch = platform.machine().lower()
system_info = ""

if "arm" in arch or "aarch64" in arch:
    # Apple Silicon (M1/M2/M3) or Raspberry Pi
    system_info = "Target Architecture: ARM64 (Apple Silicon). Use NEON intrinsics for SIMD optimization. Prefer logical cores for threading."
elif "x86" in arch or "amd64" in arch:
    # Intel i5/i7/i9 or AMD Ryzen
    system_info = "Target Architecture: x86_64 (Intel/AMD). Use AVX2/AVX-512 intrinsics for SIMD optimization."
else:
    # Fallback
    system_info = "Target Architecture: Generic. Write portable C++ code."


In [4]:
print(system_info)

Target Architecture: x86_64 (Intel/AMD). Use AVX2/AVX-512 intrinsics for SIMD optimization.


In [5]:
model=Groq()

In [46]:

system_message = f"You are a helpful assistant that reimplements Python code in high performance C++ for an {system_info}. "
system_message += "Respond only with C++ code; use comments sparingly and do not provide explanation."
system_message += "The C++ response needs to produce an identical output in the fastest possible execution time."
system_message+="You are a strict code conversion engine.\n RETURN ONLY CODE.\n WRAP CODE IN MARKDOWN: ```cpp ... ```\n NEVER write text outside the code block.\n For comments strictly use '//'\n NO conversational text (e.g., 'Here is the code', 'Explanation:').\n Start output immediately with '#include'.\n"
system_message+="Do NOT output any reasoning, thinking process, or explanations.Do NOT use conversational fillers like 'Here is the code'. Start your response immediately with ```cpp\n"
# --- THE FIX STARTS HERE ---
system_message += """
CRITICAL TYPE SAFETY RULES:
1. Python integers have arbitrary precision. C++ 'int' overflows at 2 billion.
   - ALWAYS use `int64_t` (or `long long`) for sums, accumulators, and any calculation that might exceed 32 bits.
   - NEVER use `unsigned` types (like `uint64_t` or `size_t`) for arithmetic involving subtraction (e.g., `a - b`) unless guaranteed positive. This prevents underflow wrapping.
2. If the Python code uses a Dictionary, use `std::unordered_map` instead of `std::map`.
3. If the Python code uses Lists, use `std::vector`.
"""
# --- THE FIX ENDS HERE ---


In [47]:
def user_prompt_for(python):
    user_prompt="Rewrite this python code in C++ with the fastest possible implementation that produces identical output in the least time."
    user_prompt+="Respond only with C++ code; no need to explain your work other than few comments."
    user_prompt+="Pay attention to number types to ensure not int overflow.Remember to #include all necessary C++ packages such as iomanip.\n\n"
    user_prompt+=python
    return user_prompt

In [48]:
def messages_for(python):
    return[
        {"role":"system","content":system_message},
        {"role":"user","content":user_prompt_for(python)}
    ]

In [49]:
def write_output(cpp):
    code=cpp.replace("```cpp","").replace("```","")
    with open("optimized.cpp","w") as f:
        f.write(code)


In [ ]:
# def optimize_llama(python):
#     # 3. Call the Stream
#     # LOOK: You can use 'messages=' explicitly here! No LangChain errors.
#     stream = model.chat.completions.create(
#         model="llama-3.3-70b-versatile",
#         messages=messages_for(python), # Works perfectly with your list of dicts
#         stream=True
#     )
    
#     reply = ""
    
#     # 4. Iterate (Standard OpenAI style)
#     for chunk in stream:
#         fragment = chunk.choices[0].delta.content or ""
#         reply += fragment
#         print(fragment, end='', flush=True)
        
#     write_output(reply)

In [11]:
pi="""
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
# exec(pi)

In [ ]:
# optimize_llama(pi)

In [ ]:
# !g++ -O3 -std=c++17 -march=native -o optimized.exe optimized.cpp
# !optimized.exe

In [15]:
python_hard = """
def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000        # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [ ]:
# exec(python_hard)

In [ ]:
# optimize_llama(python_hard)

In [ ]:
# !g++ -O3 -std=c++17 -march=native -o optimized.exe optimized.cpp
# !optimized.exe

In [ ]:
import re
def extract_clean_cpp(text):
    # 1. Try finding Markdown blocks first (Best case)
    pattern = r"```(?:cpp|c\+\+)?(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return max(matches, key=len).strip()
    
    # 2. Fallback: Manually slice the string
    start_idx = text.find("#include")
    if start_idx == -1:
        # If no #include, looks for other common C++ starters
        start_idx = text.find("using namespace")
    
    # If still not found, just return raw (model failed completely)
    if start_idx == -1:
        return text.strip()
        
    # 3. Find the LAST closing brace '}' to cut off trailing explanations
    # This fixes the "Wait, but in C++..." error at the end of the file
    last_brace = text.rfind("}")
    
    if last_brace != -1:
        # Return everything from the first #include to the last }
        return text[start_idx : last_brace + 1].strip()
    
    # If no braces found (weird), return from start
    return text[start_idx:].strip()

# def optimize_qwen(python):
#     # No print statements here anymore
#     full_raw_reply = ""
    
#     stream = model.chat.completions.create(
#         model="qwen/qwen3-32b",
#         messages=messages_for(python),
#         stream=True
#     )
    
#     # Silent accumulation
#     for chunk in stream:
#         fragment = chunk.choices[0].delta.content or ""
#         full_raw_reply += fragment
#         # The print statement is gone. It will just "work" in the background.

#     # Clean the result
#     clean_code = extract_clean_cpp(full_raw_reply)
    
#     # Write only the clean code
#     write_output(clean_code)

In [ ]:
# optimize_qwen(python_hard)

In [ ]:
# !g++ -O3 -std=c++17 -march=native -o optimized.exe optimized.cpp
# !optimized.exe

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 0.859036 seconds


In [59]:
def optimize_qwen(python):
    # No print statements here anymore
    full_raw_reply = ""
    
    try:
        stream = model.chat.completions.create(
            model="qwen/qwen3-32b", # Using Qwen on Groq
            messages=messages_for(python),
            stream=True
        )
        
        # Silent accumulation
        for chunk in stream:
            fragment = chunk.choices[0].delta.content or ""
            full_raw_reply += fragment
            # The print statement is gone. It will just "work" in the background.

        # Clean the result
        clean_code = extract_clean_cpp(full_raw_reply)
        
        # Write only the clean code (Yielding for Gradio)
        yield clean_code

    except Exception as e:
        yield f"// Error in optimize_qwen: {str(e)}"

def optimize_llama(python):
    # 3. Call the Stream
    try:
        stream = model.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages_for(python),
            stream=True
        )
        
        reply = ""
        
        # 4. Iterate (Standard OpenAI style)
        for chunk in stream:
            fragment = chunk.choices[0].delta.content or ""
            reply += fragment
            # print(fragment, end='', flush=True) # Commented out to prevent console spam
            
            # We yield the raw reply for Llama so the user sees it typing
            # We do a 'soft clean' just for display (remove backticks)
            yield reply.replace("```cpp", "").replace("```", "")
            
    except Exception as e:
        yield f"// Error in optimize_llama: {str(e)}"

In [61]:
import io
import sys
import subprocess
def execute_python(code):
    try:
        output = io.StringIO()
        sys.stdout = output
        exec(code)
    except Exception as e:
        return f"Error: {e}"
    finally:
        sys.stdout = sys.__stdout__
    return output.getvalue()

def execute_cpp(code):
    with open("optimized.cpp", "w") as f:
        f.write(code)
    try:
        # Windows-safe compile
        subprocess.run(["g++", "-O3", "-std=c++17", "-march=native", "-o", "optimized.exe", "optimized.cpp"], check=True, capture_output=True)
        run_cmd = ["optimized.exe"] if os.name == 'nt' else ["./optimized.exe"]
        result = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"Error:\n{e.stderr}"

def convert_router(python_code, model_choice):
    if model_choice == "Qwen (Silent & Clean)":
        yield from optimize_qwen(python_code)
    else:
        yield from optimize_llama(python_code)

In [62]:
css = """
.python {background-color: #306998;}
.cpp {background-color: #050;}
"""

with gr.Blocks(css=css, title="Python to C++ Converter") as ui:
    gr.Markdown("## 🚀 Python to C++ Optimizer")
    
    with gr.Row():
        with gr.Column():
            model_selector = gr.Dropdown(
                choices=["Qwen (Silent & Clean)", "Llama 3 (Streaming)"],
                value="Qwen (Silent & Clean)",
                label="Select Strategy"
            )
            python_input = gr.Textbox(label="Python Input:", lines=15, placeholder="Paste Python code...")
            convert_btn = gr.Button("Convert Code", variant="primary")
        
        with gr.Column():
            cpp_output = gr.Code(label="C++ Output:", language="cpp", lines=15)

    with gr.Row():
        python_run = gr.Button("Run Python")
        cpp_run = gr.Button("Run C++")
    
    with gr.Row():
        python_res = gr.TextArea(label="Python Result:", elem_classes=["python"])
        cpp_res = gr.TextArea(label="C++ Result:", elem_classes=["cpp"])

    # Connect buttons
    convert_btn.click(
        fn=convert_router, 
        inputs=[python_input, model_selector], 
        outputs=[cpp_output]
    )
    python_run.click(fn=execute_python, inputs=[python_input], outputs=[python_res])
    cpp_run.click(fn=execute_cpp, inputs=[cpp_output], outputs=[cpp_res])

if __name__ == "__main__":
    ui.launch(inbrowser=True)

C:\Users\91944\AppData\Local\Temp\ipykernel_4812\261329147.py:6: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css, title="Python to C++ Converter") as ui:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
